[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/optional-tensorflow/02_keras_cnn.ipynb)

# Convolutional Neural Networks in TensorFlow/Keras

This notebook demonstrates a CNN for image classification and introduces transfer learning with a pretrained Keras application.


## Learning objectives
- Load and preprocess image data.
- Build a CNN with `Conv2D` and `MaxPooling2D`.
- Train and evaluate a model on Fashion-MNIST.
- Sketch a transfer-learning workflow with MobileNetV2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)


## Load Fashion-MNIST
Fashion-MNIST is a drop-in replacement for MNIST with clothing categories instead of digits.


In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
X_train_full.shape


## Preprocess and reshape images
CNNs expect image tensors with an explicit channel dimension.


In [ ]:
X_valid = X_train_full[:5000].astype("float32") / 255.0
X_train = X_train_full[5000:].astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_valid = X_valid[..., np.newaxis]
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

y_valid = y_train_full[:5000]
y_train = y_train_full[5000:]

X_train.shape


## Visualize sample images
This quick check confirms the data were loaded and scaled correctly.


In [ ]:
plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()


## Build a small CNN
The model alternates convolution and pooling layers before a dense classifier head.


In [ ]:
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax")
])

cnn.summary()


## Compile and train
We use Adam optimization and sparse categorical crossentropy for the integer class labels.


In [ ]:
cnn.compile(optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

history = cnn.fit(
    X_train, y_train,
    epochs=8,
    batch_size=64,
    validation_data=(X_valid, y_valid),
    verbose=2
)


## Plot learning curves
Training and validation curves help assess model fit.


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="valid")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="valid")
plt.title("Accuracy")
plt.legend()
plt.tight_layout()


## Evaluate the CNN
Next we estimate generalization on the held-out test set.


In [ ]:
test_loss, test_acc = cnn.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


## Predict class labels
The predicted class is the index of the largest softmax probability.


In [ ]:
pred_probs = cnn.predict(X_test[:5])
pred_labels = np.argmax(pred_probs, axis=1)
print("Predicted:", [class_names[i] for i in pred_labels])
print("Actual:   ", [class_names[i] for i in y_test[:5]])


## Transfer learning overview
For more complex image problems, pretrained models often outperform small CNNs trained from scratch.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

transfer_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(160, 160, 3)),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(5, activation="softmax")
])

transfer_model.summary()


## Fine-tuning idea
After training a custom classifier head, unfreeze the top layers of the base model and continue with a smaller learning rate.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

transfer_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                       loss="sparse_categorical_crossentropy",
                       metrics=["accuracy"])


## Extension ideas
Try adding data augmentation, batch normalization, or `image_dataset_from_directory()` for a custom image dataset.
